# Investigation of `test_clean_scores_text.pkl` 

This notebook explores the content and structure of the prediction scores file generated by the CPP training script.

In [4]:
import pickle
import pandas as pd
import numpy as np
import os

file_path = '/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/cpp/decorte/job_title_desc/scores/test_clean_scores_text.pkl'

print(f"File exists: {os.path.exists(file_path)}")

File exists: True


## Loading the Data

The file is a pickle file containing a dictionary with scores and metadata.

In [5]:
with open(file_path, 'rb') as f:
    data = pickle.load(f)

print(f"Type of loaded data: {type(data)}")
if isinstance(data, dict):
    print(f"Keys: {list(data.keys())}")

Type of loaded data: <class 'dict'>
Keys: ['scores', 'target_labels', 'true_target_indices', 'split', 'histories', 'true_targets', 'job_ids']


## Understanding the Columns/Keys

Based on the training script (`train_cpp_enhanced_v2.py`), the keys are:
- **`scores`**: A similarity matrix of shape `[n_samples, n_targets]`. These are the S_text scores from the mapping network.
- **`target_labels`**: A list of all target occupation labels (all possible careers in the dataset).
- **`true_target_indices`**: The index of the actual target occupation for each sample within `target_labels`.
- **`split`**: The name of the dataset split (e.g., 'test_clean').
- **`histories`**: (If metadata was included) List of job titles in the history for each sample.
- **`true_targets`**: (If metadata was included) The label of the true target occupation for each sample.

In [3]:
if isinstance(data, dict):
    for key in data.keys():
        val = data[key]
        if hasattr(val, 'shape'):
            print(f"{key}: shape {val.shape}")
        elif isinstance(val, list):
            print(f"{key}: list of length {len(val)}")
        else:
            print(f"{key}: {val}")

scores: shape (227, 1022)
target_labels: list of length 1022
true_target_indices: list of length 227
split: test_clean
histories: list of length 227
true_targets: list of length 227


In [19]:
data['histories']

[['chef de cuisine', 'chef/general manager'],
 ['lead cook/kitchen trainer, kitchen manager',
  'lead cook',
  'galley steward (305) 436-400 (ronald strode)',
  'cook lead'],
 ['marketing communications coordinator',
  'marketing coordinator',
  'associate product manager',
  'associate category manager',
  'marketing manager and brand manager',
  'product manager'],
 ['sous chef',
  'sous chef',
  'owner',
  'garde manger/production manager',
  'regional executive chef'],
 ['sous chef',
  'executive chef',
  'executive chef',
  'chef/owner',
  'executive chef',
  'sous chef',
  'head chef'],
 ['senior business analyst',
  'last chance/one warm night',
  'pastry cook iii',
  'pastry cook ii/chocolatier',
  'pastry cook ii'],
 ['executive sous chef', 'executive chef', 'executive chef', 'executive chef'],
 ['general cook', 'general cook'],
 ['manager', 'floor management assistant'],
 ['secretary',
  'customer service rep',
  'sales associate',
  'shift lead/assistant manager',
  'sales a

## Sample Inspection

In [4]:
# Example: Look at the first sample
if isinstance(data, dict):
    idx = 0
    print(f"--- Sample {idx} ---")
    if 'histories' in data:
        print(f"History: {data['histories'][idx]}")
    if 'true_targets' in data:
        print(f"True Target Label: {data['true_targets'][idx]}")
    if 'true_target_indices' in data:
        target_idx = data['true_target_indices'][idx]
        print(f"True Target Index: {target_idx}")
        if 'target_labels' in data:
            print(f"Label from target_labels[{target_idx}]: {data['target_labels'][target_idx]}")
    
    if 'scores' in data:
        sample_scores = data['scores'][idx]
        print(f"Score for true target: {sample_scores[target_idx]:.4f}")
        top_indices = np.argsort(sample_scores)[-5:][::-1]
        print("Top 5 predicted targets:")
        for i, t_idx in enumerate(top_indices):
            label = data['target_labels'][t_idx] if 'target_labels' in data else f"Index {t_idx}"
            print(f"  {i+1}. {label} (Score: {sample_scores[t_idx]:.4f})")

--- Sample 0 ---
History: ['chef de cuisine', 'chef/general manager']
True Target Label: head chef
True Target Index: 899
Label from target_labels[899]: esco role: head chef 
 description: Head chefs manage the kitchen to oversee the preparation, cooking and service of food.
Score for true target: 0.9464
Top 5 predicted targets:
  1. esco role: head chef 
 description: Head chefs manage the kitchen to oversee the preparation, cooking and service of food. (Score: 0.9464)
  2. esco role: chef 
 description: Chefs are culinary professionals with a flair for creativity and innovation to provide a unique gastronomic experience. (Score: 0.9014)
  3. esco role: cook 
 description: Cooks are culinary operatives who are able to prepare and present food, normally in domestic and institutional environments. (Score: 0.7263)
  4. esco role: private chef 
 description: Private chefs comply with food and sanitation rules to prepare meals for their employers. They take into consideration the employer’

## Converting to DataFrame for Analysis

We can create a summary DataFrame (without the full scores matrix, which is too large).

In [5]:
if isinstance(data, dict) and 'true_targets' in data:
    df_summary = pd.DataFrame({
        'history': [" -> ".join(h) for h in data['histories']],
        'true_target': data['true_targets'],
        'true_target_idx': data['true_target_indices']
    })
    df_summary.head(10)

In [6]:
df_summary

,history,true_target,true_target_idx
0,chef de cuisine -> chef/general manager,head chef,899
1,"lead cook/kitchen trainer, kitchen manager -> ...",cook,479
2,marketing communications coordinator -> market...,brand manager,662
3,sous chef -> sous chef -> owner -> garde mange...,chef,273
4,sous chef -> executive chef -> executive chef ...,head chef,899
...,...,...,...
222,information technology consultant -> informati...,cybersecurity risk manager,235
223,correctional officer -> information technology...,computer hardware repair technician,18
224,agile project manager -> release manager -> bu...,project manager,26
225,bpo support analyst -> retail loan officer -> ...,corporate risk manager,132
